# 施密特分解深度教程

深入理解施密特分解及其在量子纠缠中的应用。

## 学习目标

1. 从数学角度理解施密特分解
2. 实现手动施密特分解
3. 理解与SVD的关系
4. 计算各种纠缠度量

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../../common')

from utils.tensor_utils import entanglement_entropy, renyi_entropy

%matplotlib inline

## 1. 施密特分解的数学

对于双分量态 $|\psi\rangle \in \mathcal{H}_A \otimes \mathcal{H}_B$:

$$
|\psi\rangle = \sum_{i,j} c_{ij} |i\rangle_A \otimes |j\rangle_B
$$

施密特分解保证存在正交基使得:

$$
|\psi\rangle = \sum_{\alpha} \lambda_\alpha |\phi_\alpha\rangle_A \otimes |\chi_\alpha\rangle_B
$$

In [ ]:
# 例子1: 贝尔态
def schmidt_decompose(psi, dim_A, dim_B):
    """
    施密特分解
    
    参数:
        psi: 态矢量 (dim_A * dim_B,)
        dim_A, dim_B: 子系统维度
    
    返回:
        lambdas, U, V: 施密特值和基
    """
    # Reshape为矩阵
    psi_matrix = psi.reshape(dim_A, dim_B)
    
    # SVD分解
    U, S, Vh = np.linalg.svd(psi_matrix, full_matrices=False)
    
    return S, U, Vh

# 贝尔态: (|00⟩ + |11⟩) / √2
bell_state = np.zeros(4)
bell_state[0] = 1/np.sqrt(2)  # |00⟩
bell_state[3] = 1/np.sqrt(2)  # |11⟩

lambdas, U, Vh = schmidt_decompose(bell_state, 2, 2)

print("贝尔态的施密特分解:")
print(f"施密特值: {lambdas}")
print(f"\n解释: 两个相等的施密特值 → 最大纠缠态")
print(f"纠缠熵: S = {entanglement_entropy(lambdas):.6f}")
print(f"理论值: S = ln(2) = {np.log(2):.6f}")

## 2. 不同纠缠态的施密特谱

In [ ]:
# 产品态 vs 纠缠态
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 产品态: |0⟩⊗|0⟩
product_state = np.zeros(4)
product_state[0] = 1.0
lambdas_prod, _, _ = schmidt_decompose(product_state, 2, 2)

axes[0].bar(range(len(lambdas_prod)), lambdas_prod, color='blue', alpha=0.7)
axes[0].set_title('Product State')
axes[0].set_ylabel('Schmidt Value')
axes[0].set_xlabel('Index')
axes[0].set_ylim([0, 1.1])

# 最大纠缠态
axes[1].bar(range(len(lambdas)), lambdas, color='red', alpha=0.7)
axes[1].set_title('Bell State (Maximally Entangled)')
axes[1].set_xlabel('Index')
axes[1].set_ylim([0, 1.1])

# 部分纠缠态: (|00⟩ + 0.5|11⟩) / normalized
partial_state = np.zeros(4)
partial_state[0] = 1.0
partial_state[3] = 0.5
partial_state = partial_state / np.linalg.norm(partial_state)
lambdas_part, _, _ = schmidt_decompose(partial_state, 2, 2)

axes[2].bar(range(len(lambdas_part)), lambdas_part, color='green', alpha=0.7)
axes[2].set_title('Partially Entangled')
axes[2].set_xlabel('Index')
axes[2].set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

print("纠缠熵对比:")
print(f"产品态: S = {entanglement_entropy(lambdas_prod):.6f}")
print(f"贝尔态: S = {entanglement_entropy(lambdas):.6f}")
print(f"部分纠缠: S = {entanglement_entropy(lambdas_part):.6f}")

## 3. GHZ态和W态（三体纠缠）

In [ ]:
# GHZ态: (|000⟩ + |111⟩) / √2
ghz_state = np.zeros(8)
ghz_state[0] = 1/np.sqrt(2)
ghz_state[7] = 1/np.sqrt(2)

# W态: (|100⟩ + |010⟩ + |001⟩) / √3
w_state = np.zeros(8)
w_state[1] = 1/np.sqrt(3)  # |001⟩
w_state[2] = 1/np.sqrt(3)  # |010⟩
w_state[4] = 1/np.sqrt(3)  # |100⟩

# 分析双分量纠缠 (A vs BC)
lambdas_ghz, _, _ = schmidt_decompose(ghz_state, 2, 4)
lambdas_w, _, _ = schmidt_decompose(w_state, 2, 4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(len(lambdas_ghz)), lambdas_ghz, color='purple', alpha=0.7)
ax1.set_title('GHZ State (A vs BC)')
ax1.set_xlabel('Schmidt Index')
ax1.set_ylabel('Schmidt Value')

ax2.bar(range(len(lambdas_w)), lambdas_w, color='orange', alpha=0.7)
ax2.set_title('W State (A vs BC)')
ax2.set_xlabel('Schmidt Index')

plt.tight_layout()
plt.show()

print("三体态纠缠分析:")
print(f"GHZ态纠缠熵: S = {entanglement_entropy(lambdas_ghz):.6f}")
print(f"W态纠缠熵: S = {entanglement_entropy(lambdas_w):.6f}")
print(f"\nGHZ态有更强的双分量纠缠！")

## 4. Rényi熵族

In [ ]:
# 计算不同阶的Rényi熵
n_values = np.linspace(0.1, 5, 50)
renyi_entropies = []

for n in n_values:
    S_n = renyi_entropy(lambdas, n=n)
    renyi_entropies.append(S_n)

plt.figure(figsize=(10, 6))
plt.plot(n_values, renyi_entropies, linewidth=2)
plt.axhline(np.log(2), color='red', linestyle='--', 
           label='von Neumann (n→1)')
plt.axvline(1, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Rényi Index $n$', fontsize=12)
plt.ylabel('Rényi Entropy $S_n$', fontsize=12)
plt.title('Rényi Entropy Family for Bell State', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("特殊情况:")
print(f"S₀ (Hartley): {renyi_entropy(lambdas, n=0.01):.6f}")
print(f"S₁ (von Neumann): {np.log(2):.6f}")
print(f"S₂ (Collision): {renyi_entropy(lambdas, n=2):.6f}")
print(f"S∞ (Min-entropy): {-np.log(max(lambdas)**2):.6f}")

## 5. 随机态的施密特谱统计

In [ ]:
# 生成随机Haar态
def random_haar_state(dim):
    """生成Haar随机态"""
    real = np.random.randn(dim)
    imag = np.random.randn(dim)
    psi = real + 1j * imag
    return psi / np.linalg.norm(psi)

# 统计100个随机态的纠缠熵
n_samples = 100
entropies_random = []

for _ in range(n_samples):
    psi_random = random_haar_state(4)
    lambdas_rand, _, _ = schmidt_decompose(psi_random, 2, 2)
    S = entanglement_entropy(lambdas_rand)
    entropies_random.append(S)

plt.figure(figsize=(10, 6))
plt.hist(entropies_random, bins=20, color='teal', alpha=0.7, edgecolor='black')
plt.axvline(np.mean(entropies_random), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {np.mean(entropies_random):.4f}')
plt.axvline(np.log(2), color='orange', linestyle=':', linewidth=2,
           label=f'Max (Bell): {np.log(2):.4f}')
plt.xlabel('Entanglement Entropy', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Entanglement Entropy for Random States', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"随机态平均纠缠熵: {np.mean(entropies_random):.6f}")
print(f"标准差: {np.std(entropies_random):.6f}")
print(f"\n大多数随机态接近最大纠缠！")

## 练习

1. 验证施密特分解的正交性
2. 计算四体纠缠态的施密特谱
3. 研究不同纠缠度量之间的关系
4. 实现negativity等其他纠缠度量

## 参考文献

- Schmidt (1907) - 原始论文
- Peres (1996) - "Separability Criterion"
- Horodecki et al. (2009) - 纠缠综述